# Gradient descent

_Machine Learning_

**Walk downhill on the cost. Each step, move opposite the slope.**

The cost $J(\theta)$ is like a landscape of hills and valleys.

     We want the lowest point (smallest cost).

---

This notebook builds gradient descent one step at a time. Run each cell, read the note above it, and watch the cost fall as each step moves opposite the slope. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — NumPy

We minimise the simplest possible cost, `J(theta) = theta^2`, whose minimum sits at `theta = 0`. We build it in three steps: (1) define the gradient, (2) run the descent loop with a sensible step size, (3) see what happens when the step size is too big.

### Step 1 — Define the cost and its gradient

Our cost is `J(theta) = theta^2`, a single smooth bowl. Its slope (derivative) is `2*theta`, which points *uphill*. We start `theta` away from the minimum at `5.0` and pick a learning rate `alpha` that controls how far each step moves.

In [ ]:
import numpy as np

# minimize J(theta) = theta^2, gradient = 2*theta, minimum at 0
def grad(theta):
    return 2 * theta

theta = 5.0       # starting point
alpha = 0.1       # learning rate (step size)

### Step 2 — Run the descent loop

Each step subtracts `alpha * grad(theta)` from `theta`, moving *opposite* the slope and so downhill. We print every fifth step to watch `theta` and the cost `J` shrink toward 0 — this is gradient descent converging.

In [ ]:
for step in range(20):
    theta = theta - alpha * grad(theta)   # step opposite the slope
    if step % 5 == 0:
        print("step %2d  theta=%.4f  J=%.4f" % (step, theta, theta ** 2))

print("converged theta:", round(theta, 5))   # close to 0

### Step 3 — Watch a too-big step size diverge

The learning rate matters. With `alpha = 1.1` each step *overshoots* the minimum and lands farther out than it started, so `theta` blows up instead of settling. This is the classic failure mode: too large a step turns descent into divergence.

In [ ]:
# too-big alpha diverges:
t = 5.0
for _ in range(5):
    t = t - 1.1 * grad(t)
print("alpha=1.1 blows up to:", round(t, 1))

## Visualize the data & results

_Fitting the diabetes BMI slope by gradient descent: does each step actually drive the cost down?_

### Step 1 — Load and centre the diabetes data

We move from the toy bowl to a real regression: predicting disease progression from BMI. We **centre** both the feature and the target (subtract their means) so the fit only needs a slope `theta` and no intercept term.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes

# real diabetes data: gradient descent on the BMI slope
db = load_diabetes()
x = db.data[:, 2]      # BMI feature
y = db.target          # disease progression

xc = x - x.mean()      # centre the feature
yc = y - y.mean()      # centre the target

### Step 2 — Descend on the slope, recording the cost

We fit a single slope `theta` for the line `y = theta * x`. The gradient of the squared-error cost is the average of `(theta*x - y) * x`. Each step nudges `theta` opposite that gradient, and we store the cost `J` after every step so we can plot the learning curve.

In [ ]:
theta = 0.0
alpha = 0.5
steps = []
costs = []

for s in range(1, 21):
    grad = np.mean((theta * xc - yc) * xc)        # slope of the cost
    theta = theta - alpha * grad                  # step opposite the slope
    cost = np.mean(0.5 * (theta * xc - yc) ** 2)  # mean squared loss at this theta
    steps.append(s)
    costs.append(cost)

### Step 3 — Plot the cost per step

We plot the recorded cost against the step number. A curve that falls and then flattens confirms gradient descent is working: each step lowers `J` until it reaches the best slope it can find.

In [ ]:
plt.plot(steps, costs, color="#ffb454", marker="o", label="J at each step")
plt.xlabel("step")
plt.ylabel("mean squared loss J")
plt.title("Gradient descent on the diabetes BMI slope")
plt.legend()
plt.show()